In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from astropy.constants import G, M_sun, c
import juliacall
from pysr import PySRRegressor
import sympy as sp
from utils import *
from models import GNN

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/Users/blu3/PycharmProjects/orbit_ml/.venv/lib/python3.9/site-packages/pysr/julia_import.py:11: UserWarning: juliacall module already imported. Make sure that you have set the environment variable `PYTHON_JULIACALL_HANDLE_SIGNALS=yes` to avoid segfaults. Also note that PySR will not be able to configure `PYTHON_JULIACALL_THREADS` or `PYTHON_JULIACALL_OPTLEVEL` for you.
  warnings.warn(


In [2]:
# Load data, y contains longitude of perihelion and x contains t, e, n, and a respectively
X, y = get_sr_ooe('OOE.csv', nrows=22505 * 1000, step=22505 // 2, scale=False)

y = (((24 * np.pi**2 * X[:,2]**2 ) / (c.value**2 * X[:,3]**3 * (1-X[:,1]**2))) * 180 * 3600 * 3600 * 24 * 365.25 + 5.3232) * X[:, 0]
y

array([ 8700.39597085,  8703.27211933,  8706.14200065, ...,
       14447.24322438, 14450.14482225, 14453.00758558])

In [ ]:

# Constants with time in years
# G_val = G.to('m^3 / (kg * yr^2)').value
# M_sun_val = M_sun.value
#c_val = c.to('m/yr').value
theoretical_precession_rate_val = 5.3232   # arcsec/yr

# Create a dictionary for the constants
constants = {
    # 'G': G_val,
    # 'M_sun': M_sun_val,
    'Pi': np.pi,
    'c': c.value,
    'W': theoretical_precession_rate_val
}

# Incorporate constants into the feature set
num_samples = X.shape[0]
X_combined = np.hstack((X, np.tile(np.array(list(constants.values())), (num_samples, 1))))
print("X data shape: ", X_combined.shape)

# Split the data
x_train, x_val, y_train, y_val = train_test_split(X_combined, y, test_size=0.2, random_state=42)

temp_dir = 'pysr_temp'
os.makedirs(temp_dir, exist_ok=True)

try:
    model = PySRRegressor(
        niterations=20000,
        ncycles_per_iteration=5000,
        binary_operators=["+", "-", "*", "/"],  # "^"],
        unary_operators=["square", "cube", "sqrt"],  # "sin", "cos", "tan", "exp"],
        constraints={
            # "^": (-1, 3),
            "sqrt": 2,
            "square": 2,
            "cube": 2,
            # "sin": 3,
            # "cos": 3,
            # "tan": 3,
            # "exp": 3,
        },
        nested_constraints={
            # "sin": {"sin": 1, "cos": 1, "tan": 1},
            # "cos": {"cos": 1, "sin": 1, "tan": 1},
            # "tan": {"tan": 1, "sin": 1, "cos": 1},
            "square": {"square": 0, "cube": 0, "sqrt": 0},
            "cube": {"square": 0, "cube": 0, "sqrt": 0},
            "sqrt": {"square": 0, "cube": 0, "sqrt": 0},
        },
        model_selection="accuracy",
        batching=True,
        batch_size=64,
        # annealing=True,
        # alpha=0.006,
        crossover_probability=0.03,
        tournament_selection_n=25,
        tournament_selection_p=0.8,
        fraction_replaced_hof=0.013,
        fraction_replaced=0.003,
        should_optimize_constants=True,
        weight_add_node=3,
        weight_insert_node=15,
        weight_delete_node=2,
        weight_mutate_operator=4,
        optimizer_nrestarts=2,
        optimize_probability=0.14,
        optimizer_iterations=4,
        parsimony=0.0000001,
        perturbation_factor=0.11,
        use_frequency=True,
        use_frequency_in_tournament=True,
        adaptive_parsimony_scaling=1,
        should_simplify=True,
        maxsize=40,
        # warmup_maxsize_by=0.02,
        maxdepth=12,
        verbosity=1,
        tempdir=temp_dir,
        temp_equation_file=True,
        delete_tempfiles=True,
        populations=48,
        population_size=146,
        procs=12,
        turbo=True,
        bumper=True,
        warm_start=True,
        # elementwise_loss="PeriodicLoss(1296000.0)",     # 360 * 3600, arcsecs in a full circle
        # cluster_manager='slurm'
    )
except Exception as e:
    print(f"Error initializing PySRRegressor: {e}")
    raise

# Fit the model
try:
    variable_names = ['t', 'e', 'a', 'T'] + list(constants.keys())
    model.fit(x_train, y_train, variable_names=variable_names)
except Exception as e:
    print(f"Error fitting the model: {e}")
    raise

# Print the best equation found
try:
    best_equation = model.sympy()
    print("Best equation found:", best_equation)
except Exception as e:
    print(f"Error getting the best equation: {e}")
    raise


# Convert the best equation to a SymPy expression
def convert_to_sympy(expression_str):
    try:
        sympy_expr = sp.sympify(expression_str)
        return sympy_expr
    except Exception as e:
        print(f"Error converting to SymPy expression: {e}")
        raise


# Substitute the actual values of constants in the best equation
def substitute_constants(equation, constants):
    for symbol, value in constants.items():
        equation = equation.subs(sp.Symbol(symbol), value)
    return equation


# Ensure the best equation is a SymPy expression
try:
    best_equation_sympy = convert_to_sympy(best_equation)
    best_equation_with_values = substitute_constants(best_equation_sympy, constants)
    print("Best equation with constant values:", best_equation_with_values)
except Exception as e:
    print(f"Error processing the best equation: {e}")
    raise

# Validate the model using the validation data
try:
    y_pred = model.predict(x_val)
except Exception as e:
    print(f"Error predicting with the model: {e}")
    raise

# Calculate the Mean Absolute Percentage Error (MAPE)
try:
    mape = np.mean(np.abs((y_val - y_pred) / y_val)) * 100
    print(f"Mean Absolute Percentage Error on validation set: {mape:.2f}%")
except Exception as e:
    print(f"Error calculating MAPE: {e}")
    raise

# Plot the original data and the model prediction
try:
    plt.scatter(x_val[:, 0], y_val, label='Validation Data')
    plt.plot(x_val[:, 0], y_pred, color='red', label='Model Prediction')
    plt.legend()
    plt.show()
except Exception as e:
    print(f"Error plotting the data: {e}")
    raise

# Save the best equation and the MAPE to a file for further evaluation
try:
    with open('pysr_evaluation.txt', 'w') as f:
        f.write(f"Best equation found: {best_equation}\n")
        f.write(f"Best equation with constant values: {best_equation_with_values}\n")
        f.write(f"Mean Absolute Percentage Error on validation set: {mape:.2f}%\n")
except Exception as e:
    print(f"Error saving the model evaluation: {e}")
    raise


X data shape:  (2001, 7)
Compiling Julia backend...


/Users/blu3/PycharmProjects/orbit_ml/.venv/lib/python3.9/site-packages/pysr/sr.py:2583: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
